# Pipeline multidataset: preparación, características y descriptor de 18 dimensiones

Tesis: Framework de selección inteligente de datos para el entrenamiento eficiente de modelos de lenguaje mediante aprendizaje por refuerzo y estimación del consumo energético. (Fase 1)

## Propósito

Construir los artefactos que recibirá la política PPO:

Hugging Face → partición sin fuga → Train/Validation/Test → 800 ejemplos de entrenamiento por dataset → GPT-2 small → perplejidad + longitud + diversidad → normalización → descriptor de 18 dimensiones → guardado reproducible.

Datasets: IMDb, AG News y TREC.

Regla metodológica: el conjunto de prueba oficial se reserva para la evaluación final. La validación se toma del conjunto de entrenamiento original y es independiente de los 800 ejemplos usados para entrenar la política.

In [ ]:
# 1. Dependencias
!pip -q install --prefer-binary -U "datasets==3.6.0" "transformers==4.57.0" "pandas>=2.2.0,<2.4.0" "numpy>=2.0.0,<2.2.0" accelerate scikit-learn pyarrow tqdm

In [ ]:
# Google drive access
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# 2. Imports y configuración
from __future__ import annotations

import html
import json
import random
import re
import time
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import datasets
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
N_TRAIN_EXAMPLES = 800
N_VAL_EXAMPLES = 300
MAX_MODEL_TOKENS = 256
FEATURE_BATCH_SIZE = 16
MAX_DATASET_CLASSES = 6
LENGTH_THRESHOLDS = (32, 64, 128, 192)
PROXY_MODEL_NAME = "openai-community/gpt2"

OUTPUT_DIR = Path('/content/drive/MyDrive/idsf-eft/framework/artifacts')
DATASET_DIR = OUTPUT_DIR / 'datasets'
FEATURE_DIR = OUTPUT_DIR / 'features'
DESCRIPTOR_DIR = OUTPUT_DIR / 'descriptors'
for d in (DATASET_DIR, FEATURE_DIR, DESCRIPTOR_DIR): d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Output:', OUTPUT_DIR)
print('datasets:', datasets.__version__)
print('transformers:', __import__('transformers').__version__)
print('torch:', torch.__version__)

Device: cuda
Output: /content/drive/MyDrive/idsf-eft/framework/artifacts
datasets: 3.6.0
transformers: 4.57.0
torch: 2.11.0+cu130


In [ ]:
# 3. Reproducibilidad
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

## 4. Registro de datasets y columnas

Todos los datasets se normalizan a las columnas comunes `text`, `label`, `dataset`, `source_split` y `source_index`. TREC usa `coarse_label`, porque la tesis trabaja con sus seis clases gruesas.

In [ ]:
DATASETS = {
    'imdb': {'hf_id':'stanfordnlp/imdb', 'text_column':'text', 'label_column':'label', 'num_classes':2, 'trust_remote_code':False},
    'ag_news': {'hf_id':'fancyzhx/ag_news', 'text_column':'text', 'label_column':'label', 'num_classes':4, 'trust_remote_code':False},
    'trec': {'hf_id':'CogComp/trec', 'text_column':'text', 'label_column':'coarse_label', 'num_classes':6, 'trust_remote_code':True},
}

# Tabla rápida de configuración
display(pd.DataFrame([{
    'dataset':k, 'hf_id':v['hf_id'], 'text':v['text_column'], 'label':v['label_column'], 'classes':v['num_classes']
} for k,v in DATASETS.items()]))

,dataset,hf_id,text,label,classes
0,imdb,stanfordnlp/imdb,text,label,2
1,ag_news,fancyzhx/ag_news,text,label,4
2,trec,CogComp/trec,text,coarse_label,6


In [ ]:
# 5. Limpieza y estandarización
HTML_TAG_RE = re.compile(r'<[^>]+>')
WS_RE = re.compile(r'\s+')

def clean_text(text: str) -> str:
    text = html.unescape(str(text))
    text = HTML_TAG_RE.sub(' ', text)
    return WS_RE.sub(' ', text).strip()

def standardize_dataset(raw_df: pd.DataFrame, dataset_name: str, text_column: str, label_column: str, source_split: str) -> pd.DataFrame:
    df = raw_df[[text_column, label_column]].copy().rename(columns={text_column:'text', label_column:'label'})
    df['text'] = df['text'].map(clean_text)
    df['label'] = pd.to_numeric(df['label'], errors='raise').astype(int)
    df['dataset'] = dataset_name
    df['source_split'] = source_split
    df['source_index'] = np.arange(len(df), dtype=np.int64)
    before = len(df)
    df = df[df['text'].str.len() > 0].reset_index(drop=True)
    if before != len(df): print(f'[{dataset_name}] eliminados {before-len(df)} textos vacíos')
    return df[['dataset','source_split','source_index','text','label']]

def class_distribution(df):
    c = df['label'].value_counts().sort_index()
    return pd.DataFrame({'label':c.index, 'count':c.values, 'proportion':(c/c.sum()).values})

In [ ]:
# 6. Carga desde Hugging Face
standardized = {}
for name, cfg in DATASETS.items():
    print(f'\n=== {name.upper()} ===')
    train_raw = load_dataset(cfg['hf_id'], split='train', trust_remote_code=cfg['trust_remote_code'])
    test_raw = load_dataset(cfg['hf_id'], split='test', trust_remote_code=cfg['trust_remote_code'])
    train_df = standardize_dataset(train_raw.to_pandas(), name, cfg['text_column'], cfg['label_column'], 'train')
    test_df = standardize_dataset(test_raw.to_pandas(), name, cfg['text_column'], cfg['label_column'], 'test')
    standardized[name] = {'train':train_df, 'test':test_df}
    print('Train original:', len(train_df))
    print('Test original :', len(test_df))
    display(class_distribution(train_df))


=== IMDB ===


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

... (16 outputs left out)

Generating train split:   0%|          | 0/5452 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Train original: 5452
Test original : 500


,label,count,proportion
0,0,86,0.015774
1,1,1250,0.229274
2,2,1162,0.213133
3,3,1223,0.224321
4,4,835,0.153155
5,5,896,0.164343


## 7. Partición experimental sin fuga

Se reservan 800 ejemplos para el entrenamiento y 300 para la validación, ambos extraídos del conjunto de entrenamiento original mediante muestreo estratificado. El conjunto de prueba oficial completo queda separado.

Esto significa que la recompensa del agente podrá usar el conjunto de validación, pero no el de prueba.

In [ ]:
def build_splits(train_df: pd.DataFrame, test_df: pd.DataFrame, n_train=800, n_val=300, seed=42):
    need = n_train + n_val
    pool, _ = train_test_split(train_df, train_size=need, stratify=train_df['label'], random_state=seed)
    train_part, val_part = train_test_split(pool, train_size=n_train, test_size=n_val, stratify=pool['label'], random_state=seed+1)
    return {
        'train': train_part.sample(frac=1, random_state=seed+2).reset_index(drop=True),
        'validation': val_part.sample(frac=1, random_state=seed+3).reset_index(drop=True),
        'test': test_df.sample(frac=1, random_state=seed+4).reset_index(drop=True),
    }

def overlap(a,b):
    A=set(zip(a['source_split'], a['source_index']))
    B=set(zip(b['source_split'], b['source_index']))
    return len(A & B)

experimental_splits={}
for name in DATASETS:
    s=build_splits(standardized[name]['train'], standardized[name]['test'], N_TRAIN_EXAMPLES, N_VAL_EXAMPLES, SEED)
    assert len(s['train'])==800 and len(s['validation'])==300
    assert overlap(s['train'], s['validation'])==0
    assert overlap(s['train'], s['test'])==0
    assert overlap(s['validation'], s['test'])==0
    experimental_splits[name]=s
    print(f'\n{name}: train={len(s["train"])}, val={len(s["validation"])}, test={len(s["test"])}')
    display(pd.concat([
        class_distribution(s['train']).assign(split='train'),
        class_distribution(s['validation']).assign(split='validation')
    ]))


imdb: train=800, val=300, test=25000


,label,count,proportion,split
0,0,400,0.5,train
1,1,400,0.5,train
0,0,150,0.5,validation
1,1,150,0.5,validation



ag_news: train=800, val=300, test=7600


,label,count,proportion,split
0,0,200,0.25,train
1,1,200,0.25,train
2,2,200,0.25,train
3,3,200,0.25,train
0,0,75,0.25,validation
1,1,75,0.25,validation
2,2,75,0.25,validation
3,3,75,0.25,validation



trec: train=800, val=300, test=500


,label,count,proportion,split
0,0,12,0.015000,train
1,1,183,0.228750,train
2,2,170,0.212500,train
3,3,180,0.225000,train
4,4,123,0.153750,train
5,5,132,0.165000,train
0,0,5,0.016667,validation
1,1,69,0.230000,validation
2,2,64,0.213333,validation
3,3,67,0.223333,validation


## 8. Modelo proxy GPT-2 small

Se utilizará `openai-community/gpt2`. Para la extracción se generan tres señales por ejemplo:

- **Perplejidad:** calculada sobre como máximo 256 tokens.
- **Longitud real:** número de tokens sin truncamiento.
- **Embedding:** promedio de los estados ocultos de la última capa en la secuencia procesada.

La diversidad se calculará posteriormente como distancia coseno respecto al centroide de los embeddings.

In [ ]:
# 9. Carga del proxy
tokenizer = AutoTokenizer.from_pretrained(PROXY_MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
proxy_model = AutoModelForCausalLM.from_pretrained(PROXY_MODEL_NAME)
proxy_model.to(DEVICE).eval()
print(PROXY_MODEL_NAME, '| hidden size:', proxy_model.config.n_embd, '|', DEVICE)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

openai-community/gpt2 | hidden size: 768 | cuda


In [ ]:
# 10. Extracción por lotes
@torch.inference_mode()
def batch_gpt2_features(texts: List[str], max_length=256):
    enc = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=max_length)
    ids, mask = enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE)
    out = proxy_model(input_ids=ids, attention_mask=mask, output_hidden_states=True, use_cache=False)

    # Pérdida causal por ejemplo -> perplejidad
    logits = out.logits[:, :-1, :].contiguous()
    labels = ids[:, 1:].contiguous()
    shift_mask = mask[:, 1:].contiguous()
    V = logits.size(-1)
    loss_tok = F.cross_entropy(logits.view(-1, V), labels.view(-1), reduction='none').view(labels.shape)
    loss_tok = loss_tok * shift_mask
    counts = shift_mask.sum(dim=1).clamp_min(1)
    nll = loss_tok.sum(dim=1) / counts
    ppl = torch.exp(nll).cpu().numpy().astype(np.float32)

    # Mean pooling de la última capa
    h = out.hidden_states[-1]
    m = mask.unsqueeze(-1).to(h.dtype)
    emb = ((h*m).sum(dim=1) / m.sum(dim=1).clamp_min(1)).cpu().numpy().astype(np.float32)
    return ppl, emb

def true_token_lengths(texts: List[str]) -> np.ndarray:
    return np.asarray([len(tokenizer(t, add_special_tokens=False, truncation=False)['input_ids']) for t in texts], dtype=np.int32)

def cosine_distance_to_centroid(emb: np.ndarray) -> np.ndarray:
    c=emb.mean(axis=0)
    en=np.linalg.norm(emb,axis=1).clip(min=1e-12)
    cn=max(float(np.linalg.norm(c)),1e-12)
    sim=(emb@c)/(en*cn)
    return (1.0-sim).astype(np.float32)

def robust_minmax(x: np.ndarray):
    x=np.asarray(x,dtype=np.float64)
    lo=float(np.percentile(x,5)); hi=float(np.percentile(x,95))
    if hi-lo < 1e-12: return np.zeros_like(x,dtype=np.float32), {'p05':lo,'p95':hi}
    return np.clip((x-lo)/(hi-lo),0,1).astype(np.float32), {'p05':lo,'p95':hi}

In [ ]:
# 11. Ejecutar características para los 800 ejemplos de train de cada dataset

def extract_features(train_df: pd.DataFrame):
    texts=train_df['text'].tolist()
    lengths=true_token_lengths(texts)
    ppls=[]; embs=[]
    t0=time.perf_counter()
    for i in tqdm(range(0,len(texts),FEATURE_BATCH_SIZE), desc=f'GPT-2 | {train_df.dataset.iloc[0]}'):
        p,e=batch_gpt2_features(texts[i:i+FEATURE_BATCH_SIZE], MAX_MODEL_TOKENS)
        ppls.append(p); embs.append(e)
    ppl=np.concatenate(ppls); emb=np.concatenate(embs)
    div=cosine_distance_to_centroid(emb)
    log_ppl=np.log(np.clip(ppl,1e-8,None)).astype(np.float32)
    n_ppl,ppl_par=robust_minmax(log_ppl)
    n_len,len_par=robust_minmax(lengths.astype(np.float64))
    n_div,div_par=robust_minmax(div.astype(np.float64))
    df=train_df.copy()
    df['perplexity']=ppl; df['log_perplexity']=log_ppl; df['true_length']=lengths; df['diversity']=div
    df['norm_log_perplexity']=n_ppl; df['norm_length']=n_len; df['norm_diversity']=n_div
    pars={'log_perplexity':ppl_par,'true_length':len_par,'diversity':div_par,'elapsed_seconds':time.perf_counter()-t0}
    return df, emb, pars

feature_tables={}; embeddings={}; selection_params={}
for name in DATASETS:
    df, emb, pars=extract_features(experimental_splits[name]['train'])
    feature_tables[name]=df; embeddings[name]=emb; selection_params[name]=pars
    np.save(FEATURE_DIR/f'{name}_proxy_embeddings.npy', emb)
    print(name, 'done')

Token indices sequence length is longer than the specified maximum sequence length for this model (1287 > 1024). Running this sequence through the model will result in indexing errors


GPT-2 | imdb:   0%|          | 0/50 [00:00<?, ?it/s]

imdb done


GPT-2 | ag_news:   0%|          | 0/50 [00:00<?, ?it/s]

ag_news done


GPT-2 | trec:   0%|          | 0/50 [00:00<?, ?it/s]

trec done


In [ ]:
# 12. Diagnóstico de características
rows=[]
for name,df in feature_tables.items():
    L=df['true_length']
    rows.append({
        'dataset':name,'n':len(df),'classes':df['label'].nunique(),
        'length_mean':L.mean(),'length_median':L.median(),'length_p95':L.quantile(.95),
        'ppl_mean':df['perplexity'].mean(),'ppl_median':df['perplexity'].median(),
        'diversity_mean':df['diversity'].mean(),'diversity_std':df['diversity'].std(ddof=1),
        'P(L>32)':(L>32).mean(),'P(L>64)':(L>64).mean(),'P(L>128)':(L>128).mean(),'P(L>192)':(L>192).mean(),
    })
diagnostics=pd.DataFrame(rows)
display(diagnostics.round(4))

,dataset,n,classes,length_mean,length_median,length_p95,ppl_mean,ppl_median,diversity_mean,diversity_std,P(L>32),P(L>64),P(L>128),P(L>192)
0,imdb,800,2,294.0688,226.5,719.05,47.091099,42.9685,0.0005,0.0005,0.9975,0.9725,0.8550,0.5925
1,ag_news,800,4,49.7350,49.0,70.00,90.348000,68.9369,0.0016,0.0015,0.9275,0.0975,0.0025,0.0000
2,trec,800,6,11.3188,11.0,20.00,394.705811,176.3505,0.0021,0.0013,0.0012,0.0000,0.0000,0.0000


## 13. Descriptor de 18 dimensiones

Para selección individual se mantienen normalizaciones dentro de cada dataset. Para el descriptor del agente se usa una normalización global sobre los datasets que participan en el entrenamiento de la política.

Esto evita que la normalización por dataset borre precisamente la información que interesa al agente, como la diferencia entre textos cortos y textos largos.

Las 18 dimensiones son:

**12:** media, desviación estándar, P25 y P75 de `log_perplexity`, `true_length` y `diversity`.

**2:** número de clases normalizado por 6 y entropía de clase normalizada por `log(6)`.

**4:** proporciones de ejemplos con longitud >32, >64, >128 y >192 tokens.

In [ ]:
# 14. Scaler global de las características
GLOBAL_FEATURES=['log_perplexity','true_length','diversity']

def fit_global_scaler(feature_tables):
    all_df=pd.concat([df[GLOBAL_FEATURES] for df in feature_tables.values()], ignore_index=True)
    s={}
    for f in GLOBAL_FEATURES:
        x=all_df[f].to_numpy(float); lo=float(np.percentile(x,5)); hi=float(np.percentile(x,95))
        if hi-lo<1e-12: raise ValueError(f'Rango degenerado en {f}')
        s[f]={'p05':lo,'p95':hi}
    return s

def apply_global_scaler(df, scaler):
    out=df.copy()
    for f in GLOBAL_FEATURES:
        lo,hi=scaler[f]['p05'],scaler[f]['p95']
        out[f'global_norm_{f}']=np.clip((out[f].astype(float)-lo)/(hi-lo),0,1).astype(np.float32)
    return out

global_scaler=fit_global_scaler(feature_tables)
for name in feature_tables:
    feature_tables[name]=apply_global_scaler(feature_tables[name], global_scaler)
display(pd.DataFrame(global_scaler).T)

,p05,p95
log_perplexity,3.208405,6.357017
true_length,7.000000,468.050000
diversity,0.000223,0.003731


In [ ]:
# 15. Construcción del descriptor de 18D
DESCRIPTOR_COLUMNS=[
 'ppl_mean','ppl_std','ppl_p25','ppl_p75',
 'length_mean','length_std','length_p25','length_p75',
 'div_mean','div_std','div_p25','div_p75',
 'num_classes_norm','class_entropy_norm',
 'p_gt_32','p_gt_64','p_gt_128','p_gt_192'
]

def normalized_entropy(labels, max_classes=6):
    p=labels.value_counts(normalize=True).to_numpy(float)
    H=float(-(p*np.log(p+1e-12)).sum())
    return float(np.clip(H/np.log(max_classes),0,1))

def build_descriptor(df):
    ppl=df['global_norm_log_perplexity'].to_numpy(float)
    L=df['global_norm_true_length'].to_numpy(float)
    div=df['global_norm_diversity'].to_numpy(float)
    rawL=df['true_length'].to_numpy(float)
    d={
      'ppl_mean':ppl.mean(),'ppl_std':ppl.std(ddof=1),'ppl_p25':np.percentile(ppl,25),'ppl_p75':np.percentile(ppl,75),
      'length_mean':L.mean(),'length_std':L.std(ddof=1),'length_p25':np.percentile(L,25),'length_p75':np.percentile(L,75),
      'div_mean':div.mean(),'div_std':div.std(ddof=1),'div_p25':np.percentile(div,25),'div_p75':np.percentile(div,75),
      'num_classes_norm':df['label'].nunique()/6.0,'class_entropy_norm':normalized_entropy(df['label'],6),
      'p_gt_32':np.mean(rawL>32),'p_gt_64':np.mean(rawL>64),'p_gt_128':np.mean(rawL>128),'p_gt_192':np.mean(rawL>192),
    }
    d={k:float(np.clip(v,0,1)) for k,v in d.items()}
    return pd.Series(d,index=DESCRIPTOR_COLUMNS,dtype=np.float32)

descriptors={name:build_descriptor(df) for name,df in feature_tables.items()}
descriptor_df=pd.DataFrame(descriptors).T
descriptor_df.index.name='dataset'
assert descriptor_df.shape==(3,18)
assert ((descriptor_df>=0)&(descriptor_df<=1)).all().all()
display(descriptor_df.round(4))

,ppl_mean,ppl_std,ppl_p25,ppl_p75,length_mean,length_std,length_p25,length_p75,div_mean,div_std,div_p25,div_p75,num_classes_norm,class_entropy_norm,p_gt_32,p_gt_64,p_gt_128,p_gt_192
dataset,,,,,,,,,,,,,,,,,,
imdb,0.1815,0.1192,0.0926,0.2546,0.5418,0.2807,0.3205,0.7792,0.0938,0.1413,0.0118,0.1112,0.3333,0.3869,0.9975,0.9725,0.8550,0.5925
ag_news,0.3332,0.2117,0.1719,0.4794,0.0927,0.0303,0.0737,0.1084,0.3551,0.2232,0.1895,0.4634,0.6667,0.7737,0.9275,0.0975,0.0025,0.0000
trec,0.6060,0.2910,0.3914,0.8509,0.0097,0.0093,0.0022,0.0130,0.4956,0.2765,0.2687,0.6995,1.0000,0.9211,0.0012,0.0000,0.0000,0.0000


In [ ]:
# 16. Guardado de todos los artefactos necesarios para la fase 2
for name,df in feature_tables.items():
    df.to_parquet(FEATURE_DIR/f'{name}_train800_features.parquet', index=False)
    for split,part in experimental_splits[name].items():
        part.to_parquet(DATASET_DIR/f'{name}_{split}.parquet', index=False)

descriptor_df.to_csv(DESCRIPTOR_DIR/'dataset_descriptors_18d.csv')

with open(DESCRIPTOR_DIR/'global_feature_scaler.json','w',encoding='utf-8') as f:
    json.dump(global_scaler,f,indent=2,ensure_ascii=False)
with open(FEATURE_DIR/'selection_normalizers.json','w',encoding='utf-8') as f:
    json.dump(selection_params,f,indent=2,ensure_ascii=False)

metadata={
 'seed':SEED,'n_train_examples':N_TRAIN_EXAMPLES,'n_val_examples':N_VAL_EXAMPLES,'max_model_tokens':MAX_MODEL_TOKENS,
 'feature_batch_size':FEATURE_BATCH_SIZE,'proxy_model':PROXY_MODEL_NAME,'device':str(DEVICE),
 'datasets':DATASETS,'descriptor_columns':DESCRIPTOR_COLUMNS,
 'package_versions': {'datasets':datasets.__version__, 'transformers':__import__('transformers').__version__, 'torch':torch.__version__},
}
with open(OUTPUT_DIR/'pipeline_metadata.json','w',encoding='utf-8') as f:
    json.dump(metadata,f,indent=2,ensure_ascii=False)

for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file(): print(p.relative_to(OUTPUT_DIR))

datasets/ag_news_test.parquet
datasets/ag_news_train.parquet
datasets/ag_news_validation.parquet
datasets/imdb_test.parquet
datasets/imdb_train.parquet
datasets/imdb_validation.parquet
datasets/trec_test.parquet
datasets/trec_train.parquet
datasets/trec_validation.parquet
descriptors/dataset_descriptors_18d.csv
descriptors/global_feature_scaler.json
features/ag_news_proxy_embeddings.npy
features/ag_news_train800_features.parquet
features/imdb_proxy_embeddings.npy
features/imdb_train800_features.parquet
features/selection_normalizers.json
features/trec_proxy_embeddings.npy
features/trec_train800_features.parquet
pipeline_metadata.json
